In [171]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import FunctionTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyRegressor
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

from sklearn.linear_model import LinearRegression
from sklearn.compose import TransformedTargetRegressor
from sklearn.model_selection import cross_validate
from sklearn.base import clone
from sklearn.model_selection import KFold, cross_val_score


In [158]:
df = pd.read_csv("../Data/processed/smartphones_clean.csv")

print("Dataset shape:", df.shape)
print(df.head())
print(df.info())

Dataset shape: (1020, 31)
                      model  \
0              OPPO F35 Pro   
1  Samsung Galaxy S25 Ultra   
2             OPPO K14 Plus   
3   Motorola Edge 70 Fusion   
4               Vivo V80 5G   

                                                 url     brand scrape_date  \
0                             https://smpx.to/jifOik      OPPO  2026-10-14   
1  https://www.smartprix.com/mobiles/samsung-gala...   Samsung  2026-10-14   
2  https://www.smartprix.com/mobiles/oppo-k14-plu...      OPPO  2026-10-14   
3  https://www.smartprix.com/mobiles/motorola-edg...  Motorola  2026-10-14   
4  https://www.smartprix.com/mobiles/vivo-v80-5g-...      Vivo  2026-10-14   

   price_inr  spec_score                 processor_name  cpu_cores  cpu_ghz  \
0      45999          80             Dimensity 7360 Max        8.0     2.50   
1      94999          92  Snapdragon 8 Elite for Galaxy        8.0     4.47   
2      25999          75             Dimensity 7360 Max        8.0     2.50   
3 

In [159]:
# Select only the records intended for the primary analysis
ml_df = df.loc[df["in_analysis_scope"] == True].copy()

# Define the target
target = "price_inr"

print("ML dataset shape:", ml_df.shape)
print("Missing target values:", ml_df[target].isna().sum())
print("Target price summary:")
print(ml_df[target].describe())

ML dataset shape: (838, 31)
Missing target values: 0
Target price summary:
count       838.000000
mean      44113.464200
std       46450.424966
min        6499.000000
25%       18999.000000
50%       29999.000000
75%       49990.000000
max      449900.000000
Name: price_inr, dtype: float64


In [160]:
# Target: smartphone listed price in INR
ml_df["price_inr"] = pd.to_numeric(ml_df["price_inr"], errors="coerce")
print("ML rows:", len(ml_df))
print("Missing prices:", ml_df["price_inr"].isna().sum())
print(ml_df["price_inr"].describe())

assert ml_df["price_inr"].notna().all()

ML rows: 838
Missing prices: 0
count       838.000000
mean      44113.464200
std       46450.424966
min        6499.000000
25%       18999.000000
50%       29999.000000
75%       49990.000000
max      449900.000000
Name: price_inr, dtype: float64


In [161]:
# Numeric predictors
numeric_features = [
    "spec_score",
    "ram_gb",
    "storage_gb",
    "cpu_cores",
    "cpu_ghz",
    "battery_mah",
    "fast_charge_w",
    "screen_in",
    "res_short",
    "res_long",
    "refresh_hz",
    "rear_main_mp",
    "rear_mp_value_count",
    "front_mp",
    "os_major"
]

# Categorical predictors
categorical_features = [
    "brand",
    "processor_name",
    "os_family",
    "charging_status",
    "is_foldable",
    "is_5g",
    "has_nfc",
    "has_esim",
    "has_memory_card"
]




In [162]:
# Select features
features = numeric_features + categorical_features

X = ml_df[features].copy()
y = ml_df["price_inr"].copy()

print("X shape:", X.shape)
print("Features:", len(features))

X shape: (838, 24)
Features: 24


In [163]:
print("Numeric features:", len(numeric_features))
print("Categorical features:", len(categorical_features))
print("Total features:", len(features))

Numeric features: 15
Categorical features: 9
Total features: 24


In [164]:

X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.20,random_state=42)

print("\nTraining rows:", len(X_train))
print("Testing rows:", len(X_test))


Training rows: 670
Testing rows: 168


In [165]:
#  Preprocess numeric and categorical features separately
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(
        strategy="constant",
        fill_value="Unknown"
    )),
    ("string_converter", FunctionTransformer(
        lambda X: X.astype(str),
        feature_names_out="one-to-one"
    )),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])
preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])

In [166]:

# 4. Baseline ML model: Linear Regression

model = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", LinearRegression())
])

# Train the model
model.fit(X_train, y_train)

# Predict prices for unseen test data
predictions = model.predict(X_test)

# Evaluate the baseline model
mae = mean_absolute_error(y_test, predictions)
rmse = np.sqrt(mean_squared_error(y_test, predictions))
r2 = r2_score(y_test, predictions)

print("BASELINE MODEL: LINEAR REGRESSION")
print("-" * 40)
print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))
print("Features:", len(features))
print()
print(f"MAE:  ₹{mae:,.2f}")
print(f"RMSE: ₹{rmse:,.2f}")
print(f"R²:   {r2:.4f}")


BASELINE MODEL: LINEAR REGRESSION
----------------------------------------
Training rows: 670
Testing rows: 168
Features: 24

MAE:  ₹6,818.77
RMSE: ₹10,651.11
R²:   0.9600


In [167]:

# 5. Dummy baseline: predict the training-set mean

dummy_model = DummyRegressor(strategy="mean")

dummy_model.fit(X_train, y_train)
dummy_predictions = dummy_model.predict(X_test)

dummy_mae = mean_absolute_error(y_test, dummy_predictions)
dummy_r2 = r2_score(y_test, dummy_predictions)

print("DUMMY BASELINE")
print("-" * 40)
print(f"MAE: ₹{dummy_mae:,.2f}")
print(f"R²:  {dummy_r2:.4f}")


DUMMY BASELINE
----------------------------------------
MAE: ₹31,384.12
R²:  -0.0087


In [168]:

# 6. Five-fold cross-validation

cv = KFold(n_splits=5, shuffle=True, random_state=42)

cv_scores = cross_val_score(model, X,y, cv=cv, scoring="r2", n_jobs=-1)

print("CROSS-VALIDATION RESULTS")
print("-" * 40)
print("Fold scores:", cv_scores.round(4))
print(f"Mean R²: {cv_scores.mean():.4f}")
print(f"Minimum R²: {cv_scores.min():.4f}")
print(f"Maximum R²: {cv_scores.max():.4f}")


CROSS-VALIDATION RESULTS
----------------------------------------
Fold scores: [0.96   0.9252 0.914  0.9327 0.9356]
Mean R²: 0.9335
Minimum R²: 0.9140
Maximum R²: 0.9600


In [169]:

# 7. Helper function for feature ablation

def evaluate_ablation(feature_to_remove):

    selected_features = [
        col for col in features
        if col != feature_to_remove
    ]

    num_cols = [
        col for col in numeric_features
        if col in selected_features
    ]

    cat_cols = [
        col for col in categorical_features
        if col in selected_features
    ]

    ablation_preprocessor = ColumnTransformer([
        ("numeric", numeric_pipeline, num_cols),
        ("categorical", categorical_pipeline, cat_cols)
    ])

    ablation_model = Pipeline([
        ("preprocessor", ablation_preprocessor),
        ("regressor", LinearRegression())
    ])

    scores = cross_val_score(
        ablation_model,
        X[selected_features],
        y,
        cv=cv,
        scoring="r2",
        n_jobs=-1
    )

    return scores


In [170]:

# 8. Compare both feature ablations

ablation_results = []

for feature_to_remove in ["spec_score", "os_major"]:

    ablated_scores = evaluate_ablation(feature_to_remove)

    # Compare matching folds
    differences = cv_scores - ablated_scores

    print(f"\nWithout {feature_to_remove}")
    print("Ablated scores:", ablated_scores.round(4))
    print("Base - ablated:", differences.round(4))
    print(f"Mean ablated R²: {ablated_scores.mean():.4f}")
    print(f"Mean difference: {differences.mean():.4f}")

    for fold, (base, ablated, diff) in enumerate(
        zip(cv_scores, ablated_scores, differences),
        start=1
    ):
        ablation_results.append({
            "Feature removed": feature_to_remove,
            "Fold": fold,
            "Base R²": base,
            "Ablated R²": ablated,
            "Base - ablated": diff
        })

ablation_df = pd.DataFrame(ablation_results)

print("\nPer-fold comparison:")
print(ablation_df.round(4).to_string(index=False))



Without spec_score
Ablated scores: [0.9566 0.9116 0.911  0.9218 0.934 ]
Base - ablated: [0.0035 0.0137 0.003  0.011  0.0016]
Mean ablated R²: 0.9270
Mean difference: 0.0065

Without os_major
Ablated scores: [0.9601 0.9255 0.9124 0.9327 0.9355]
Base - ablated: [-0.0001 -0.0003  0.0016 -0.      0.0001]
Mean ablated R²: 0.9333
Mean difference: 0.0003

Per-fold comparison:
Feature removed  Fold  Base R²  Ablated R²  Base - ablated
     spec_score     1   0.9600      0.9566          0.0035
     spec_score     2   0.9252      0.9116          0.0137
     spec_score     3   0.9140      0.9110          0.0030
     spec_score     4   0.9327      0.9218          0.0110
     spec_score     5   0.9356      0.9340          0.0016
       os_major     1   0.9600      0.9601         -0.0001
       os_major     2   0.9252      0.9255         -0.0003
       os_major     3   0.9140      0.9124          0.0016
       os_major     4   0.9327      0.9327         -0.0000
       os_major     5   0.9356      0


### I expect the log-price model may have a lower R² on the original price scale because it focuses more on proportional errors than large absolute errors. I expect MAPE to improve because it evaluates prediction errors relative to actual prices, although this is not guaranteed.###

In [172]:

# Evaluate models on the original price scale
scoring = {
    "r2": "r2",
    "mae": "neg_mean_absolute_error",
    "mape": "neg_mean_absolute_percentage_error"
}

print("Metrics configured.")
print("R²: higher is better")
print("MAE: lower is better")
print("MAPE: lower is better")


Metrics configured.
R²: higher is better
MAE: lower is better
MAPE: lower is better


In [173]:

# Original price target
plain_results = cross_validate(clone(model), X, y, cv=cv, scoring=scoring, n_jobs=-1)

print("Original-price model evaluated.")
print(f"Mean CV R²: {plain_results['test_r2'].mean():.4f}")
print(f"Mean CV MAE: ₹{-plain_results['test_mae'].mean():,.2f}")
print(f"Mean CV MAPE: {-plain_results['test_mape'].mean() * 100:.2f}%")


Original-price model evaluated.
Mean CV R²: 0.9335
Mean CV MAE: ₹7,310.01
Mean CV MAPE: 21.41%


In [174]:

# Log-transformed target
log_model = TransformedTargetRegressor(
    regressor=clone(model),
    func=np.log1p,
    inverse_func=np.expm1
)

log_results = cross_validate(log_model, X,y, cv=cv,scoring=scoring, n_jobs=-1)

print("Log-price model evaluated.")
print(f"Mean CV R²: {log_results['test_r2'].mean():.4f}")
print(f"Mean CV MAE: ₹{-log_results['test_mae'].mean():,.2f}")
print(f"Mean CV MAPE: {-log_results['test_mape'].mean() * 100:.2f}%")


Log-price model evaluated.
Mean CV R²: 0.9263
Mean CV MAE: ₹6,231.93
Mean CV MAPE: 13.80%


In [176]:

comparison = pd.DataFrame([
    {
        "Model": "Original price",
        "Mean R²": plain_results["test_r2"].mean(),
        "Mean MAE (INR)": -plain_results["test_mae"].mean(),
        "Mean MAPE (%)": -plain_results["test_mape"].mean() * 100
    },
    {
        "Model": "Log price",
        "Mean R²": log_results["test_r2"].mean(),
        "Mean MAE (INR)": -log_results["test_mae"].mean(),
        "Mean MAPE (%)": -log_results["test_mape"].mean() * 100
    }
])

print(comparison.round(4).to_string(index=False))

print("\nFold-by-fold R²:")
print(pd.DataFrame({
    "Fold": range(1, len(plain_results["test_r2"]) + 1),
    "Original price": plain_results["test_r2"],
    "Log price": log_results["test_r2"]
}).round(4).to_string(index=False))


         Model  Mean R²  Mean MAE (INR)  Mean MAPE (%)
Original price   0.9335       7310.0120        21.4140
     Log price   0.9263       6231.9254        13.8039

Fold-by-fold R²:
 Fold  Original price  Log price
    1          0.9600     0.9552
    2          0.9252     0.9419
    3          0.9140     0.8991
    4          0.9327     0.8987
    5          0.9356     0.9369
